In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models, applications
from tensorflow.keras.utils import to_categorical

# 1. Load CIFAR-10 data
print("Loading data...")
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

# 2. Preprocess the data
# VGG16 expects specific preprocessing.
# Also, we'll use a subset to speed up the lab demo.
x_train = tf.keras.applications.vgg16.preprocess_input(x_train[:5000])
x_test = tf.keras.applications.vgg16.preprocess_input(x_test[:1000])
y_train = to_categorical(y_train[:5000], 10)
y_test = to_categorical(y_test[:1000], 10)

# 3. Load the Pre-trained VGG16 Model
# include_top=False means we leave out the final 1000-class classifier
base_model = applications.VGG16(weights='imagenet',
                                include_top=False,
                                input_shape=(32, 32, 3))

# 4. Freeze the base model
# This ensures we don't destroy the pre-learned ImageNet weights
base_model.trainable = False

# 5. Build the Final Model
model = models.Sequential([
    base_model,
    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(10, activation='softmax')
])

# 6. Compile and Train
model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

print("\nStarting Transfer Learning training...")
model.fit(x_train, y_train, epochs=5, validation_data=(x_test, y_test))

# 7. Final Evaluation
loss, acc = model.evaluate(x_test, y_test)
print(f"\nTransfer Learning Test Accuracy: {acc*100:.2f}%")

Loading data...
170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 5s 0us/step
58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step

Starting Transfer Learning training...
Epoch 1/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 10s 30ms/step - accuracy: 0.3570 - loss: 8.6440 - val_accuracy: 0.4960 - val_loss: 2.5200
Epoch 2/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.4828 - loss: 2.3862 - val_accuracy: 0.4990 - val_loss: 1.7871
Epoch 3/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.5430 - loss: 1.5834 - val_accuracy: 0.5410 - val_loss: 1.6051
Epoch 4/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.5956 - loss: 1.2565 - val_accuracy: 0.5270 - val_loss: 1.5556
Epoch 5/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.6506 - loss: 1.0428 - val_accuracy: 0.5540 - val_loss: 1.5049
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.5540 - loss: 1.5049

Transfer Learning Test Accuracy: 55.40%
